# Time discretization schemes (Euler, Milstein, exact GBM)

How **approximate schemes** relate to the **exact GBM** stepper exposed in Python, and how to study **accuracy vs cost** when you can only tune **`num_steps`** and **`num_paths`** at the high level.

## Concept

**Euler–Maruyama** and related schemes approximate SDEs on a grid; error usually scales with the step size for smooth payoffs. **Exact GBM** sampling matches the **one-step lognormal** law over each interval, so **European** prices under GBM should not exhibit discretization bias in \( \Delta t \) when the engine uses exact transitions. The Python **`McEngine`** and **`EuropeanPricer`** paths for GBM are wired to **`ExactGbm`** on the Rust side.

Scheme selectors (Euler–Maruyama, log-Euler, Milstein, exact GBM) live purely inside the Rust crate `finstack-monte-carlo` and are not exposed as Python handles today. Python users study **accuracy vs cost** only through `num_paths` and `num_steps`.

## API walkthrough

Discretisation schemes are Rust-internal; the Python surface exposes `McEngine` and `EuropeanPricer` which use the exact GBM stepper automatically. Verify that only these high-level entry points are available on `finstack.monte_carlo`.

In [ ]:
import finstack.monte_carlo as mc

print("Public Python MC entry points for GBM pricing:")
for name in ("McEngine", "EuropeanPricer", "PathDependentPricer", "LsmcPricer"):
    cls = getattr(mc, name)
    print(f"  {name}: {cls!r}")

print(
    "\nScheme handles (ExactGbm, EulerMaruyama, LogEuler, Milstein) are "
    "Rust-internal and intentionally not exposed as Python classes."
)


## Examples

Use **`McEngine`** with different **`TimeGrid`** resolutions for the same European call: with **exact GBM** the mean should track **Black–Scholes** regardless of **`num_steps`**, while **wall time** grows with more steps. Then sweep **`num_paths`** to separate **Monte Carlo noise** from **discretization** (which is absent here for Europeans).

Tables: ATM inputs; **step count** mainly affects **runtime** under **exact GBM**, while **path count** controls **standard error** and **distance to the analytical** price.

In [ ]:
import time
from finstack.monte_carlo import McEngine, TimeGrid, black_scholes_call

spot, strike, rate, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
bs = black_scholes_call(spot, strike, rate, q, vol, T)
print(f"Black–Scholes call (anchor): {bs:.6f}")

step_counts = [8, 32, 128, 512]
print("num_steps | MC_mean | stderr | |error vs BS| | sec")
for n in step_counts:
    tg = TimeGrid(t_max=T, num_steps=n)
    eng = McEngine(num_paths=40_000, time_grid=tg, seed=7)
    t0 = time.perf_counter()
    res = eng.price_european_call(spot, strike, rate, q, vol)
    elapsed = time.perf_counter() - t0
    m = res.mean.amount
    err = abs(m - bs)
    print(f"{n:9d} | {m:.6f} | {res.stderr:.6f} | {err:.6f} | {elapsed:.4f}")


In [ ]:
from finstack.monte_carlo import EuropeanPricer, black_scholes_call

spot, strike, rate, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
bs = black_scholes_call(spot, strike, rate, q, vol, T)
path_counts = [1_000, 5_000, 10_000, 50_000, 100_000]
print("num_paths | MC_mean | stderr | |error vs BS|")
for n in path_counts:
    pr = EuropeanPricer(num_paths=n, seed=99)
    r = pr.price_call(spot, strike, rate, q, vol, T, num_steps=252)
    m = r.mean.amount
    print(f"{n:9d} | {m:.6f} | {r.stderr:.6f} | {abs(m - bs):.6f}")


## Takeaways

- Prefer **exact transitions** for GBM when available: **no discretization bias** for state dynamics between grid points. The Python engine does this for you.
- **Euler / log-Euler / Milstein** live in the Rust crate only. Approximate schemes are reached via Rust glue, not a Python switch on `McEngine`.
- For production ATM Europeans, tune **`num_paths`** (and **variance reduction** elsewhere) before chasing ultra-fine **`num_steps`** under exact GBM.
- When you *do* use approximate schemes (other models), expect a **bias vs step size** trade-off: smaller **\( \Delta t \)** costs more CPU for the same **paths**.